In [44]:
import pandas as pd

In [45]:
CATALOGUE_PATH = "../data/processed/metadata/segregatedMergedCatalogue.txt"

BAD_BUBBLES_PATH = "../data/processed/metadata/badBubbles.txt"

OUTPUT_PATH = "../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt"


In [46]:
df = pd.read_csv(CATALOGUE_PATH)

print("Total bubbles before cleaning:", len(df))
df.head()

Total bubbles before cleaning: 891


,GLOBAL_ID,ID,RA_DMS,DEC_DMS,SEMI_MAJ_PC,SEMI_MIN_PC,AVG_RAD_PC,PA_DEG,ARM,DIST_ARM_PC,GAL_RAD_KPC,CATALOGUE
0,1,1,24d08m55.9s,15d48m7.9s,86,86,86,0,3,145,5.22,A
1,3,3,24d08m57.0s,15d48m9.0s,42,42,42,0,3,138,5.21,A
2,10,10,24d08m59.3s,15d48m6.1s,209,171,196,0,3,112,5.04,A
3,12,12,24d09m0.4s,15d47m52.2s,163,93,139,20,3,80,4.62,A
4,13,13,24d09m0.4s,15d48m2.6s,50,50,50,0,3,95,4.90,A


In [47]:
bad_df = pd.read_csv(BAD_BUBBLES_PATH)

print("Bad bubbles listed:", len(bad_df))
bad_df.head()

Bad bubbles listed: 120


,GLOBAL_ID
0,26
1,110
2,255
3,282
4,2345


In [48]:
# df housekeeping

df.columns = df.columns.str.strip()
df["GLOBAL_ID"] = df["GLOBAL_ID"].astype(int)

bad_df.columns = bad_df.columns.str.strip()
bad_df["GLOBAL_ID"] = bad_df["GLOBAL_ID"].astype(int)

bad_df_unique = bad_df.drop_duplicates(subset=["GLOBAL_ID"])

print("Unique bad bubbles:", len(bad_df_unique))

# checking bad bubbles matching df_all
matched_bad = bad_df_unique.merge(
    df["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="inner"
)

unmatched_bad = bad_df_unique.merge(
    df["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="left",
    indicator=True
)

unmatched_bad = unmatched_bad[unmatched_bad["_merge"] == "left_only"]

print("Matching bad bubbles:", len(matched_bad))
print("Unmatched bad bubbles:", len(unmatched_bad))

Unique bad bubbles: 108
Matching bad bubbles: 108
Unmatched bad bubbles: 0


In [49]:
df_clean = df.merge(
    bad_df_unique["GLOBAL_ID"],
    on=["GLOBAL_ID"],
    how="left",
    indicator=True
)

df_clean = df_clean[df_clean["_merge"] == "left_only"].drop(columns=["_merge"])

print("Total bubbles before cleaning:", len(df))
print("Total bubbles after cleaning:", len(df_clean))
print("Removed bubbles:", len(df) - len(df_clean))

Total bubbles before cleaning: 891
Total bubbles after cleaning: 783
Removed bubbles: 108


In [50]:
df_clean.to_csv(OUTPUT_PATH, index=False)

print("Saved full collective catalogue to", OUTPUT_PATH)

Saved full collective catalogue to ../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt
